# Suite CDEF — C definitions, read from the store

The names a C translation unit declares, in the scopes and name spaces C gives them (`C.Definitions.define(store,
path)`), read from the unit the store holds and computed on demand, nothing stored: one entity per name and kind per
scope, merging a function's prototypes with its definition, a variable's `extern` declarations with its definition
and a tag's declaration with its body, and lookup outward that keeps C's name spaces apart (ordinary identifiers,
tags, labels, each struct's members) and the preprocessor's macros. A place is an `Anchor`, plain data that outlives
the transaction it was read in: its statement's place below the unit, the statement's property and the term's path
within it, written as text (`body[1].body.body[0]:declarators[0].declarator.name`), by which each case names the
places it checks. The whole tree is values, statements within terms included: a statement expression's statements
are its term's `block`, a struct's member declarations its specifier's `members`, a conditional directive's lines its
`group`, each anchored at the term's place followed by the statement places within
(`body[0]:specifiers[0].members.members[1]`). Each case parses a unit into one store (`C.Parse.parse`), defines it,
and checks the entities, their declarations, and what names refer to. The cases port 0.3's C cases of DEF
(`05_Definitions.ipynb`), with C's own rules where 0.3 read C as C++.

In [ ]:
import itertools
import time
from pathlib import Path

from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import C
from mbse.Programs.C import Definitions as D, Parse, Statements
from mbse.Programs.Framework.Model import WORKSPACE
from mbse.Schemas.Framework import Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = C.store()
PATHS = (f"case{n}.c" for n in itertools.count())


def define(text):
    """The unit `text` parses to, written in the store at a path of its own."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return D.define(STORE, path)


def written(*body):
    """The unit whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.C.Statement`, as `Parse.parse` writes one, at a path of its own."""
    path = next(PATHS)

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "unit": txn.Programs.C.TranslationUnit})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.unit)}),
                lambda _: _.unit.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return D.define(STORE, path)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def refers(program, at):
    return [f"{e.kind}:{e.qualified_name()}" for e in D.referents(program, at)]


def named(program, spelling):
    """The places a name spelled `spelling` is written, as text, in source order."""
    return [str(at) for at in program.named(spelling)]

## CDEF-01 · What declarations declare, and which merge: one entity per name and kind per scope, and which declaration defines it

In [ ]:
program = define("""
int f(int x);
int f(int x) { return x; }
extern int shared;
int shared = 1;
extern int elsewhere;
int tentative;
int tentative;
typedef unsigned long size, *sizes;
struct point;
struct point { int x, y; };
union value { int i; float f; } initial;
enum color : unsigned char { RED, GREEN = RED + 1, };
enum mode : int;
int (*pointer)(int);
static const char *names[4], *(*table)[3];
""")
assert listing(program) == [
    ("function", "f", 2), ("parameter", "x", 1), ("parameter", "f.x", 1), ("variable", "shared", 2),
    ("variable", "elsewhere", 1), ("variable", "tentative", 2), ("type alias", "size", 1), ("type alias", "sizes", 1),
    ("struct", "point", 2), ("field", "point.x", 1), ("field", "point.y", 1), ("union", "value", 1),
    ("field", "value.i", 1), ("field", "value.f", 1), ("variable", "initial", 1), ("enumeration", "color", 1),
    ("enumerator", "RED", 1), ("enumerator", "GREEN", 1), ("enumeration", "mode", 1), ("variable", "pointer", 1),
    ("variable", "names", 1), ("variable", "table", 1)]
f = program.lookup("f")[0]
assert f.declarations == ["body[0]:declarators[0].declarator.declarator.name", "body[1]:declarator.declarator.name"]
assert f.definition == "body[1]:declarator.declarator.name" and program.entity_of("body[1]") is f  # not the prototype
assert f.scope.kind == "function" and f.scope.node == "body[1]" and f.scope.owner is f
assert program.entity_of("body[0]:declarators[0]") is f  # an init-declarator declares what its name does
prototype = program.scope_of("body[0]:declarators[0].declarator.parameters[0].declarator.name")
assert prototype.kind == "prototype" and prototype.lookup("x")[0].definition is not None  # its own `x`
shared = program.lookup("shared")[0]
assert shared.definition == "body[3]:declarators[0].declarator.name"  # not the `extern` declaration
assert program.lookup("elsewhere")[0].definition is None  # declared, defined elsewhere
assert program.lookup("tentative")[0].definition == "body[5]:declarators[0].declarator.name"  # the first, tentative
point = program.lookup("point")[0]
assert point.declarations == ["body[8]:specifiers[0].name", "body[9]:specifiers[0].name"]
assert point.definition == "body[9]:specifiers[0].name" and program.entity_of("body[9]:specifiers[0]") is point
assert point.scope.kind == "struct" and point.scope.node == "body[9]:specifiers[0].members"
assert program.lookup("point.y")[0].definition == (
    "body[9]:specifiers[0].members.members[0]:declarators[1].declarator.name")
assert program.lookup("RED")[0].parent is program.root  # an enumerator is the enclosing scope's ordinary identifier
assert refers(program, "body[11]:specifiers[0].enumerators[1].value.left") == ["enumerator:RED"]
assert program.lookup("mode")[0].definition is None  # `enum mode : int;` declares it, without its enumerators
assert program.lookup("pointer")[0].kind == "variable"  # a pointer to a function is an object
assert program.root.separator == "." and program.root.kind == "file"
print(len(listing(program)), "entities")

## CDEF-02 · Scopes: a function's parameters, body and labels in one scope; blocks, `for` statements, prototypes and statement expressions

In [ ]:
program = define("""
int total;
int f(int n, int a[static n]) {
    int total = n;
    for (int i = 0; i < n; i++) { int j = i; total += j; }
    {
        int total = 0;
        (void)total;
    }
    if (n) { int local = ({ int t = n; t; }); }
again:
    if (total) goto again;
    asm goto("jmp %l0" : : : : again, done);
done:
    return total + i;
}
void g(void (*callback)(int code), int code);
int h = ({ outside: 1; });
""")
f = program.lookup("f")[0]
assert [(e.kind, e.name) for e in f.scope.entities()] == [
    ("parameter", "n"), ("parameter", "a"), ("variable", "total"), ("label", "again"), ("label", "done")]
scopes = {e.qualified_name(): e.parent.kind for e in program.entities()}
assert (scopes["i"], scopes["j"], scopes["local"], scopes["t"]) == ("block", "block", "block", "block")
assert program.scope_of("body[1]:declarator.parameters[1].declarator.size") is f.scope  # `a[static n]` sees `n`
assert refers(program, "body[1]:declarator.parameters[1].declarator.size") == ["parameter:f.n"]
total = named(program, "total")
assert total[1] == "body[1].body.body[0]:declarators[0].declarator.name"  # the body's outermost block: the function's
assert refers(program, total[-1]) == ["variable:f.total"] and refers(program, total[4]) == ["variable:total"]
assert total[4] == "body[1].body.body[2].body[1]:expression.operand"  # the inner block's own
assert refers(program, "body[1].body.body[6]:value.right") == []  # `i`, the loop's
assert program.scope_of("body[1].body.body[1]").kind == "function"  # the loop, in the function's scope
assert program.scope_of("body[1].body.body[1].initializer").kind == "block"  # its declaration, in its own
statement = "body[1].body.body[3].consequence.body[0]:declarators[0].initializer"
assert program.anchor(f"{statement}.block").statement == f"{statement}.block"  # the block, at the term's place
assert refers(program, f"{statement}.block.body[0]:declarators[0].initializer") == ["parameter:f.n"]
assert refers(program, f"{statement}.block.body[1]:expression") == ["variable:t"]
assert program.scope_of(f"{statement}.block.body[0]").node == f"{statement}.block"
assert refers(program, "body[1].body.body[4].statement.consequence:label") == ["label:f.again"]  # the `goto`
assert program.space_of("body[1].body.body[4].statement.consequence:label") == "label"
assert named(program, "done") == ["body[1].body.body[5]:labels[1]", "body[1].body.body[6]:label"]
assert refers(program, "body[1].body.body[5]:labels[1]") == ["label:f.done"]  # an `asm goto`'s labels
assert program.entity_of("body[1].body.body[4]") is program.lookup("f.again")[0]  # the labeled statement declares it
code = program.named("code")
assert [program.scope_of(at).kind for at in code] == ["prototype", "prototype"] and program.scope_of(
    code[0]) is not program.scope_of(code[1])  # each prototype's own
assert [e.kind for e in program.lookup("g")] == ["function"] and program.lookup("g")[0].scope is None
assert program.lookup("outside")[0].parent is program.root  # a label outside every function, in GNU's block: the file's
print("ok")

## CDEF-03 · Name spaces: ordinary identifiers, tags, labels and members apart, each name found only in its own, GNU's `&&label` a label's; and macros

In [ ]:
program = define("""
#define LIMIT 10
#define LIMIT 11
#define MAX(a, b) ((a) > (b) ? (a) : (b))
struct node { struct node *next; int node; } node;
typedef struct node node_t;
int node_count(node_t *head) {
    struct node *p = head;
    int n = 0;
node:
    if (p) { n = MAX(n, LIMIT); p = p->next; goto node; }
    return n + p->node;
}
#if defined(LIMIT) && LIMIT > 5 || UNDEFINED
int limited;
#endif
#include MAX_HEADER
""")
nodes = program.named("node")
assert [(program.space_of(at), refers(program, at)) for at in nodes] == [
    (None, ["struct:node"]), ("tag", ["struct:node"]), (None, ["field:node.node"]), (None, ["variable:node"]),
    ("tag", ["struct:node"]), ("tag", ["struct:node"]), (None, ["label:node_count.node"]),
    ("label", ["label:node_count.node"])]
assert nodes[1] == "body[3]:specifiers[0].members.members[0]:specifiers[0].name"  # within its own body
assert named(program, "next") == ["body[3]:specifiers[0].members.members[0]:declarators[0].declarator.declarator.name"]
# `p->next` and `p->node`: a member's name depends on the type, so it is not visited
assert refers(program, "body[5].body.body[2].statement.consequence.body[1]:expression.right.object") == [
    "variable:node_count.p"]
assert refers(program, named(program, "node_t")[1]) == ["type alias:node_t"]
assert program.space_of(named(program, "node_t")[1]) == "ordinary"
assert D.MEANINGS["struct"] == "tag" and D.MEANINGS["field"] == "member" and D.MEANINGS["macro"] == "macro"
# Macros: in the program's own scope, a name in a condition or an `#include` theirs; elsewhere found where no ordinary
# entity of the name is
limit = program.macros.lookup("LIMIT")[0]
assert limit.kind == "macro" and limit.declarations == ["body[0]:directive.name", "body[1]:directive.name"]
assert limit.definition == "body[0]:directive.name" and program.entity_of("body[0]:directive") is limit
assert program.lookup("LIMIT") == [] and program.scope_of("body[0]:directive.name") is program.root
assert [program.space_of(at) for at in program.named("LIMIT")[2:]] == ["ordinary", "macro", "macro"]
assert [refers(program, at) for at in program.named("LIMIT")[2:]] == [["macro:LIMIT"]] * 3
assert refers(program, named(program, "MAX")[1]) == ["macro:MAX"]  # `MAX(n, LIMIT)`
undefined = named(program, "UNDEFINED")[0]
assert program.space_of(undefined) == "macro" and refers(program, undefined) == []
assert program.space_of(named(program, "MAX_HEADER")[0]) == "macro"
assert named(program, "a") == [] and named(program, "b") == []  # a macro's parameters name nothing but in its text
# GNU's labels as values: `&&label` a label's name, found from any block of its function, as a `goto`'s; a variable of
# the name apart; an asm label and a `_Pragma` name nothing
program = define("""
int done;
void *jump(int i) {
    static void *table[] = {&&done, &&again};
    register int r asm("r1") = i;
    _Pragma("GCC unroll 2")
    { return &&done; }
again:
done:
    return 0;
}
""")
assert [(program.space_of(at), refers(program, at)) for at in program.named("done")] == [
    (None, ["variable:done"]), ("label", ["label:jump.done"]), ("label", ["label:jump.done"]),
    (None, ["label:jump.done"])]
assert named(program, "done")[2] == "body[1].body.body[3].body[0]:value.label"
assert program.scope_of(named(program, "done")[2]).kind == "block"  # found from a block, in its function's scope
assert refers(program, named(program, "again")[0]) == ["label:jump.again"]
assert [e.name for e in program.lookup("jump")[0].scope.entities()] == ["i", "table", "r", "again", "done"]
assert program.anchor("body[1].body.body[1]:declarators[0].declarator.asm") is not None
assert named(program, "r1") == [] and program.anchor("body[1].body.body[2]") is not None  # the pragma's directive
print("ok")

## CDEF-04 · Structs and unions: members in a scope of their own, anonymous members their enclosing struct's, tags and enumerators in the enclosing scope, and what names nothing

In [ ]:
program = define("""
struct outer {
    struct inner { int depth; } in;
    union { int i; struct { float f; }; };
    struct { int x; } named;
    enum { LOW, HIGH = LOW + 2 } level;
    unsigned flags : WIDTH;
    int : 0;
    int sized[SIZE];
};
struct outer o = { .in.depth = 1, .i = 2, [0] = 3 };
int use(struct outer *p) { return p->in.depth + o.named.x + sizeof(struct inner); }
struct inner { long again; };
""")
outer = program.lookup("outer")[0]
assert [(e.kind, e.name) for e in outer.scope.entities()] == [
    ("field", "in"), ("field", "named"), ("field", "level"), ("field", "flags"), ("field", "sized")]
assert outer.scope.lookup("i")[0].kind == "field" and program.lookup("outer.f")[0].kind == "field"  # anonymous
assert program.lookup("outer.x") == [] and program.lookup("outer.depth") == []  # a named member's own
assert program.lookup("inner")[0].parent is program.root  # C has no scope of a struct's: its tags are the enclosing's
assert [e.kind for e in program.lookup("LOW") + program.lookup("HIGH")] == ["enumerator", "enumerator"]
assert refers(program, named(program, "LOW")[1]) == ["enumerator:LOW"]
assert program.scope_of(named(program, "WIDTH")[0]) is program.root  # a bit-field's width, where the struct stands
assert program.scope_of(named(program, "SIZE")[0]) is program.root
unnamed = program.anchor("body[0]:specifiers[0].members.members[5]:declarators[0]")
assert program.scope_of(unnamed) is outer.scope and program.entity_of(unnamed) is None  # `int : 0`, unnamed
assert named(program, "depth") == [
    "body[0]:specifiers[0].members.members[0]:specifiers[0].members.members[0]:declarators[0].declarator.name"]
assert named(program, "i") == [  # a designator's member names nothing
    "body[0]:specifiers[0].members.members[1]:specifiers[0].members.members[0]:declarators[0].declarator.name"]
assert program.anchor("body[1]:declarators[0].initializer.items[0].designators[0]") is not None
assert program.anchor("body[1]:declarators[0].initializer.items[0].designators[0].name") is None
assert refers(program, named(program, "inner")[1]) == ["struct:inner"]  # `sizeof(struct inner)`
assert [e.kind for e in program.entities() if e.name is None] == ["union", "struct", "struct", "enumeration"]
assert program.scope_of("body[0]:specifiers[0].members.members[1]").kind == "struct"
inner = program.lookup("inner")[0]  # defined twice, which C rejects: one entity, its members its first body's
assert len(inner.declarations) == 2 and inner.definition == inner.declarations[0]
assert [e.name for e in inner.scope.entities()] == ["depth"]
assert refers(program, named(program, "again")[0]) == ["field:inner.again"]  # its second body's own scope
print("ok")

## CDEF-05 · Directives: both branches of a conditional bind, each in the scope the directive stands in; groups among statements, members and enumerators

In [ ]:
program = define("""
#ifdef WIDE
typedef long word;
#elif defined(NARROW)
typedef short word;
#else
typedef int word;
#endif
struct packet {
#if VERSION > 1
    word checksum;
#else
    word legacy;
#endif
};
enum flag {
    NONE,
#ifdef EXTRA
    EXTRA_FLAG,
#endif
};
int f(void) {
#ifndef QUIET
    int loud = 1;
#endif
    return loud;
}
""")
word = program.lookup("word")[0]
assert word.declarations == ["body[0]:directive.group.body[0]:declarators[0].declarator.name",
                             "body[0]:directive.alternative.group.body[0]:declarators[0].declarator.name",
                             "body[0]:directive.alternative.alternative.group.body[0]:declarators[0].declarator.name"]
assert word.definition == word.declarations[0] and word.parent is program.root
assert program.anchor("body[0]:directive.group").statement == "body[0]:directive.group"  # the group's place
assert program.scope_of("body[0]:directive.group.body[0]") is program.root
packet = program.lookup("packet")[0]
assert [e.name for e in packet.scope.entities()] == ["checksum", "legacy"]  # a group's members, the struct's
assert program.lookup("packet.checksum")[0].definition == (
    "body[1]:specifiers[0].members.members[0]:directive.group.members[0]:declarators[0].declarator.name")
assert refers(program, named(program, "word")[3]) == ["type alias:word"]  # the member's type
assert program.space_of(named(program, "VERSION")[0]) == "macro"
assert program.lookup("EXTRA_FLAG")[0].definition == "body[2]:specifiers[0].enumerators[1].items[0].name"
assert program.lookup("EXTRA_FLAG")[0].parent is program.root  # an enumerator, in a directive's items
assert refers(program, "body[3].body.body[1]:value") == ["variable:f.loud"]  # in the function's scope
assert [program.space_of(at) for at in program.named("NARROW") + program.named("QUIET")] == ["macro", "macro"]
print("ok")

## CDEF-06 · Linkage and old-style definitions: a block's `extern` and function declarations declare the file's entity; K&R declarations declare the parameters

In [ ]:
program = define("""
int counter;
int helper(int);
int old(a, b)
    int a;
    char *b;
{
    extern int counter;
    extern int later;
    int helper(int);
    static int kept;
    return a + counter + helper(b[0]) + later;
}
int later = 2;
""")
assert listing(program) == [
    ("variable", "counter", 2), ("function", "helper", 2), ("function", "old", 1), ("parameter", "old.a", 2),
    ("parameter", "old.b", 2), ("variable", "later", 2), ("variable", "old.kept", 1)]
counter = program.lookup("counter")[0]
assert counter.declarations == ["body[0]:declarators[0].declarator.name",
                                "body[2].body.body[0]:declarators[0].declarator.name"]
assert counter.parent is program.root and counter.definition == counter.declarations[0]
assert program.scope_of(counter.declarations[1]).kind == "function"  # written in the block, found from it
later = program.lookup("later")[0]
assert later.declarations == ["body[2].body.body[1]:declarators[0].declarator.name",
                              "body[3]:declarators[0].declarator.name"]
assert later.definition == "body[3]:declarators[0].declarator.name"
assert len(program.lookup("helper")) == 1 and program.lookup("helper")[0].definition is None
assert refers(program, "body[2].body.body[4]:value.left.left.left") == ["parameter:old.a"]
assert refers(program, "body[2].body.body[4]:value.right") == ["variable:later"]
assert program.lookup("old.kept")[0].kind == "variable"  # `static`: its own, not linked
a = program.lookup("old.a")[0]
assert a.declarations == ["body[2]:declarator.parameters[0].declarator.name",
                          "body[2].declarations[0]:declarators[0].declarator.name"]
assert program.anchor("body[2].declarations[1]").statement == "body[2].declarations[1]"  # a K&R declaration's place
assert program.scope_of("body[2].declarations[1]") is program.lookup("old")[0].scope
print("ok")

## CDEF-07 · Every entity of the conformance source, every name located, and what each declarator binds

Read independently of `define`, by a walk of the stored unit's value: every statement, declaration, member
declaration and error, and every part holding them (a member list, a group, a block a statement expression holds), through
`Terms.kind_of` and `Objects.properties`, is located, its anchor's text its place in the unit's value, and the anchors
come in the walk's order, which is source order. Every other place is located, unless it is where a name names
nothing (a member's name, an `asm` operand's name, a macro's parameters): the walk does not descend there, and the
pairs of kind and property where it stops are listed. Every identifier and variable located has its name written
there, and each declarator's identifier, but in a type name, declares an entity of its name.

In [ ]:
HOLDERS = {f"{Statements.LANGUAGE}.{name}"
           for name in (*Statements.KINDS, "Error", "MemberList", "Group", "MemberGroup")}
LISTS = {f"{Statements.LANGUAGE}.{name}" for name in ("Body", "Members", "Declarations")}


def record_of(value):
    """The record a value holds through unions, and its kind: a term's, or its schema's name in lower case."""
    while not Terms.is_term(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        ((_, value),) = Objects.properties(value).items()
    return Terms.kind_of(value) if Terms.is_term(value) else (value.as_schema().name.rsplit(".", 1)[-1].lower(), value)


def holds(record, name, value):
    """Whether a property holds statement places: statements, member declarations, errors, a body, a member list or a
    group."""
    if record.as_schema().properties[name].schema.name in LISTS:
        return True
    return hasattr(value, "as_schema") and not Terms.is_term(value) and record_of(value)[1].as_schema().name in HOLDERS


def conformance(source):
    """The program of a conformance source, every place of it checked; the walk's counts, and where it stops."""
    STORE.transaction(Parse.parse((SOURCES / source).read_text(encoding="utf-8"), source))
    started = time.perf_counter()
    program = D.define(STORE, source)
    elapsed = time.perf_counter() - started
    anchors, unlocated, counts = [], set(), {"names": 0, "declarators": 0}

    def walk(record, path):
        anchor = program.anchor(path or "<unit>")
        assert program.scope_of(anchor) is not None and anchor.property is None and anchor.statement == path, path
        anchors.append(anchor)
        for name, value in Objects.properties(record).items():
            if not holds(record, name, value):
                parts(value, f"{path}:{name}", (record_of(record)[0], name), False)
                continue
            for i, element in enumerate(value.values) if isinstance(value, Objects.Listed) else [(None, value)]:
                step = name if i is None else f"{name}[{i}]"
                walk(record_of(element)[1], f"{path}.{step}" if path else step)

    def parts(value, place, context, typed):
        if isinstance(value, Objects.Listed):
            for i, element in enumerate(value.values):
                parts(element, f"{place}[{i}]", context, typed)
            return
        if not hasattr(value, "as_schema"):
            return
        tag, record = record_of(value)
        if record.as_schema().name in HOLDERS:  # a block, a member list or a group a term holds
            walk(record, place)
            return
        if program.scope_of(place) is None:
            unlocated.add(context)
            return
        fields = Objects.properties(record)
        if tag in ("identifier", "variable"):
            counts["names"] += 1
            assert program.name_at(place) == fields.get("spelling", fields.get("name")), place
        if tag == "iddeclarator" and not typed:
            counts["declarators"] += 1
            assert program.entity_of(f"{place}.name").name == program.name_at(f"{place}.name"), place
        for name, held in fields.items():
            parts(held, f"{place}.{name}", (tag, name), typed or tag == "typeid")

    with STORE.transaction():
        walk(STORE.singleton(WORKSPACE).files({"path": source}), "")
    assert program.anchor("<unit>") == program.root.node == "<unit>"
    assert [a.order for a in anchors] == sorted(a.order for a in anchors)  # the walk's order is source order
    return program, len(anchors), counts, unlocated, elapsed


program, statements, counts, unlocated, elapsed = conformance("c23.c")
kinds = {}
for entity in [*program.entities(), *program.macros.entities()]:
    kinds[entity.kind] = kinds.get(entity.kind, 0) + 1
assert set(kinds) == set(D.MEANINGS), set(D.MEANINGS) - set(kinds)
assert (statements, counts) == (214, {"names": 241, "declarators": 94}), (statements, counts)
assert unlocated == {("memberexpression", "member"), ("fielddesignator", "name"), ("asmoperand", "name"),
                     ("definedirective", "parameters")}, unlocated
assert program.lookup("point.i")[0].kind == "field"  # an anonymous union's
assert program.macros.lookup("TRACE")[0].declarations[2] == (
    "body[11]:directive.alternative.alternative.group.body[0]:directive.name")  # each branch's
print(sum(kinds.values()), "entities;", statements, "statement places,", counts, f"defined in {elapsed:.2f} s")

## CDEF-08 · Anchors, and the transaction `define` reads in

An anchor is plain data: its text, its parts and its order, read with no transaction open; the text of a statement's
anchor is its place in the unit's value. `define` joins no transaction: it opens its own, and `define_file` reads
within the caller's; a path the workspace holds no file at is a LookupError.

In [ ]:
program = define("""
int f(int a) {
    return a;
}
int g = ({ a; });
""")
use = program.named("a")[1]
assert (use.statement, use.property, use.term) == ("body[0].body.body[0]", "value", ()) and str(use) == (
    "body[0].body.body[0]:value")
assert repr(use) == "Anchor('body[0].body.body[0]:value')" and use == program.anchor(str(use)) and use != 1
assert hash(use) == hash("body[0].body.body[0]:value") and use in {"body[0].body.body[0]:value"}
parameter = program.named("a")[0]
assert (parameter.property, parameter.term) == ("declarator", ("parameters", 0, "declarator", "name"))
assert parameter.order < use.order
within = program.named("a")[2]  # within a statement expression's block
assert (within.statement, within.property, within.term) == ("body[1]:declarators[0].initializer.block.body[0]",
                                                            "expression", ())
assert program.root.node == "<unit>" and program.anchor("body[0].body").statement == "body[0].body"
with STORE.transaction():
    file = STORE.singleton(WORKSPACE).files({"path": program.path})
    function = Objects.properties(file)["body"].values[0]  # body[0]
    kind = Terms.kind_of(function)[0]
    joined = D.define_file(STORE, file)  # within the caller's transaction, from the unit's visitor
    with raises(ValueError, match="transactions do not nest"):  # define opens its own: called with none open
        D.define(STORE, program.path)
assert kind == "functiondefinition" and program.scope_of("body[0].body.body[0]") is program.lookup("f")[0].scope
assert listing(joined) == listing(program) == [("function", "f", 1), ("parameter", "f.a", 1), ("variable", "g", 1)]
assert D.referents(joined, use) == [joined.lookup("f.a")[0]]  # an anchor of one program finds its place in another
assert program.lookup("a", at=use) == program.lookup("f.a") and refers(program, within) == []
with raises(LookupError, match="absent.c"):
    D.define(STORE, "absent.c")
assert D.referents(program, "body[9]:value") == [] and program.anchor("body[9]:value") is None
print("ok")

## CDEF-09 · Invalid syntax: an `Error` binds nothing itself, the statements recovered within it bind in its scope, wherever it stands; an `error` term's recovered terms are uses, and a `missing` term binds nothing

In [ ]:
program = define("""int a = 1;
int f(void) {
    int x = = 1;
    if (a { int inner = a; }
    return inner;
}
struct s { int = ; int ok; };
int g = (a + );
""")
assert listing(program) == [("variable", "a", 1), ("function", "f", 1), ("variable", "f.inner", 1), ("struct", "s", 1),
                            ("field", "s.ok", 1), ("variable", "g", 1)]
assert program.anchor("body[1].body.body[0]") is not None and named(program, "x") == []  # its own text binds nothing
assert program.lookup("f.inner")[0].definition == "body[1].body.body[1].body[0]:declarators[0].declarator.name"
assert program.lookup("f.inner")[0].parent.kind == "function"  # recovered, in its scope
assert refers(program, "body[1].body.body[1].body[0]:declarators[0].initializer") == ["variable:a"]
member = program.anchor("body[2]:specifiers[0].members.members[0]")  # an `Error` where a member declaration stands
assert member is not None and program.scope_of(member) is program.lookup("s")[0].scope
assert named(program, "a")[-1] == "body[3]:declarators[0].initializer.expression.left"
assert program.anchor("body[3]:declarators[0].initializer.expression.right") is not None  # `missing`: no name
assert program.name_at("body[3]:declarators[0].initializer.expression.right") is None
# What C rejects but the schemas hold, written: a struct with neither a name nor a body declares nothing, and a name in
# a type name's declarator, which C's grammar has none of, is a use
INT = lambda p: p.primitivetypespecifier(lambda k: k.keyword("int"))  # noqa: E731
program = written(
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("struct"))])),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.sizeofexpression(lambda z: z.operand(
        lambda o: o.typeid(lambda t: t.specifiers([INT]).declarator(lambda d: d.iddeclarator(lambda i: i.name(
            lambda n: n.identifier(lambda x: x.spelling("x")))))))))))
assert listing(program) == [] and program.scope_of("body[0]:specifiers[0]") is program.root
assert named(program, "x") == ["body[1]:expression.operand.declarator.name"]
assert program.space_of("body[1]:expression.operand.declarator.name") == "ordinary"
print("ok")